# Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# Load environment variables (OPENAI_API_KEY, OPENAI_BASE_URL, TAVILY_API_KEY)
# load_dotenv() reads the .env file in the current working directory (project/starter)
load_dotenv()

True

### VectorDB Instance

In [5]:
# Instantiate your ChromaDB Client
# Persist to a local "chromadb" directory so Part 2 can reuse the same collection
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# Pick one embedding function.
# We use OpenAI embeddings (matching lib/vector_db.py's VectorStoreManager).
# The api_key is passed explicitly; api_base is left to the underlying openai.OpenAI()
# client, which automatically reads OPENAI_BASE_URL from the environment (Vocareum proxy).
# NOTE: use this same embedding function when loading the collection in Part 2.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [7]:
# Create a collection
# get_or_create_collection makes re-running the notebook idempotent (mirrors
# VectorStoreManager.get_or_create_store in lib/vector_db.py).
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    # upsert (instead of add) so re-running this cell against the persistent
    # collection updates existing documents rather than raising on duplicate IDs
    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

print(f"Indexed {collection.count()} games into collection '{collection.name}'")

Indexed 15 games into collection 'udaplay'


### Semantic Search

With the documents embedded and stored, you can query the collection using natural language. ChromaDB embeds the query with the same embedding function and returns the most semantically similar games (ranked by distance — lower is closer).

In [9]:
# Semantic search demonstration
# Query the collection with natural-language questions (no exact keyword match
# required) to confirm the embeddings + vector DB return relevant games.
demo_queries = [
    "realistic racing simulator with lots of cars",
    "first 3D Mario platformer",
    "Nintendo crossover fighting game",
]

for q in demo_queries:
    results = collection.query(query_texts=[q], n_results=3)
    print(f"Query: {q!r}")
    for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
        print(
            f"  - {meta['Name']} "
            f"[{meta['Platform']}, {meta['YearOfRelease']}] "
            f"(distance={dist:.4f})"
        )
    print()

Query: 'realistic racing simulator with lots of cars'
  - Gran Turismo [PlayStation 1, 1997] (distance=0.1216)
  - Gran Turismo 5 [PlayStation 3, 2010] (distance=0.1294)
  - Mario Kart 8 Deluxe [Nintendo Switch, 2017] (distance=0.2012)



Query: 'first 3D Mario platformer'
  - Super Mario 64 [Nintendo 64, 1996] (distance=0.1058)
  - Super Mario World [Super Nintendo Entertainment System (SNES), 1990] (distance=0.1343)
  - Mario Kart 8 Deluxe [Nintendo Switch, 2017] (distance=0.1867)



Query: 'Nintendo crossover fighting game'
  - Super Smash Bros. Melee [GameCube, 2001] (distance=0.1386)
  - Marvel's Spider-Man [PlayStation 4, 2018] (distance=0.1890)
  - Mario Kart 8 Deluxe [Nintendo Switch, 2017] (distance=0.1936)

